# ครั้งที่ 4 — Fixed-point (การแทนทศนิยมแบบจุดตรึง)
**51423364 Embedded System Programming | เรียนรู้ด้วยตนเอง 31 ก.ค. 2569**

MCU หลายรุ่นไม่มีหน่วยคำนวณทศนิยม (FPU) การใช้ `float` จึงช้าและกินหน่วยความจำ
วิธีแก้คือเก็บทศนิยมด้วย **จำนวนเต็ม** แล้วตกลงกันว่า "บิตล่าง n ตัวคือส่วนทศนิยม" — นั่นคือ fixed-point

> วิธีใช้: กด `Shift + Enter` รันทีละเซลล์ แล้วลองแก้ตัวเลขดูผล


## 1. แนวคิด Qm.n
`Q8.8` = 16 บิต แบ่งเป็นส่วนจำนวนเต็ม 8 บิต + ส่วนทศนิยม 8 บิต (มีเครื่องหมาย)

In [2]:
N = 8   # จำนวนบิตทศนิยม (Q8.8)

def to_q(x, n=N):
    """ทศนิยม -> fixed-point (จำนวนเต็ม)"""
    return round(x * (1 << n))      # คูณด้วย 2^n

def from_q(v, n=N):
    """fixed-point -> ทศนิยม"""
    return v / (1 << n)             # หารด้วย 2^n

print("3.75 ใน Q8.8 =", to_q(3.75))          # 960
print("960 กลับเป็นทศนิยม =", from_q(960))    # 3.75
print("บิต:", format(to_q(3.75), '016b'))

3.75 ใน Q8.8 = 960
960 กลับเป็นทศนิยม = 3.75
บิต: 0000001111000000


**อ่านบิตให้ออก:** `0000001111000000`
- 8 บิตบน `00000011` = 3 (ส่วนจำนวนเต็ม)
- 8 บิตล่าง `11000000` = 192/256 = 0.75 (ส่วนทศนิยม)

## 2. ช่วงค่าและความละเอียด

In [3]:
for name, n, bits in [("Q4.4", 4, 8), ("Q8.8", 8, 16), ("Q16.16", 16, 32)]:
    lo = from_q(-(1 << (bits-1)), n)
    hi = from_q((1 << (bits-1)) - 1, n)
    step = from_q(1, n)
    print(f"{name:8s} ช่วง {lo:12.4f} ถึง {hi:14.6f} | ละเอียด {step}")

Q4.4     ช่วง      -8.0000 ถึง       7.937500 | ละเอียด 0.0625
Q8.8     ช่วง    -128.0000 ถึง     127.996094 | ละเอียด 0.00390625
Q16.16   ช่วง  -32768.0000 ถึง   32767.999985 | ละเอียด 1.52587890625e-05


**สังเกต trade-off:** เพิ่มบิตทศนิยม (n) → ละเอียดขึ้น แต่ช่วงค่าแคบลง
เลือก Q ให้เหมาะกับงาน เช่น อุณหภูมิ −40..125°C ใช้ Q8.8 พอ แต่ตำแหน่ง GPS ต้องละเอียดกว่า

### ลองเอง 1
จงหาว่า `Q4.4` (8 บิต) เก็บค่าได้มากสุดเท่าใด และเลข 6.25 เก็บเป็นค่าอะไร

In [4]:
print(to_q(6.25, 4))                  # ?
print(format(to_q(6.25, 4), '08b'))   # ดูบิต

100
01100100


## 3. บวก / ลบ — ทำตรง ๆ ได้เลย (ถ้า Q เท่ากัน)

In [5]:
a = to_q(2.5)    # 640
b = to_q(1.5)    # 384
s = a + b        # บวกจำนวนเต็มธรรมดา
print(a, "+", b, "=", s, "->", from_q(s))   # 4.0 ถูกต้อง

640 + 384 = 1024 -> 4.0


## 4. คูณ — ต้องเลื่อนบิตขวา n ครั้ง
เพราะผลคูณของสองค่าที่มี n บิตทศนิยม จะได้ **2n** บิตทศนิยม

In [6]:
a = to_q(2.5)     # 640
b = to_q(1.5)     # 384
raw = a * b       # ผลดิบ = 245760 (มี 16 บิตทศนิยม!)
print("ผลดิบ (ผิด) :", from_q(raw), "<- มากเกินไป 256 เท่า")

result = raw >> N          # เลื่อนขวา 8 บิต = หารด้วย 256
print("หลัง >> 8   :", result, "->", from_q(result))   # 3.75 ถูกต้อง

ผลดิบ (ผิด) : 960.0 <- มากเกินไป 256 เท่า
หลัง >> 8   : 960 -> 3.75


### ลองเอง 2
จงคำนวณ `1.25 × 4.0` ใน Q8.8 ให้ถูกต้อง (อย่าลืม `>> N`)

In [7]:
x = to_q(1.25)
y = to_q(4.0)
ans = (x * y) >> N
print(from_q(ans))   # ควรได้ 5.0

5.0


## 5. Rounding error — ราคาที่ต้องจ่าย
ไม่ใช่ทุกเลขทศนิยมจะเก็บได้พอดี

In [ ]:
val = 0.1
q = to_q(val)
back = from_q(q)
print(f"0.1 -> Q8.8 = {q} -> กลับได้ {back}")
print(f"คลาดเคลื่อน = {abs(val - back):.8f}")

# บวกสะสม 1000 ครั้ง
acc = 0
for _ in range(1000):
    acc += q
print("บวก 0.1 จำนวน 1000 ครั้ง ควรได้ 100 แต่ได้:", from_q(acc))

0.1 -> Q8.8 = 26 -> กลับได้ 0.1015625
คลาดเคลื่อน = 0.00156250
บวก 0.1 จำนวน 1000 ครั้ง ควรได้ 100 แต่ได้: 101.5625


**บทเรียน:** ความคลาดเคลื่อนเล็ก ๆ สะสมได้ — ในงานจริงต้องเลือก n ให้พอ หรือชดเชยความคลาดเคลื่อน

### ลองเอง 3
ลองเปลี่ยน `N` เป็น 12 แล้วรันเซลล์บนใหม่ ดูว่าความคลาดเคลื่อนลดลงแค่ไหน

In [26]:
val = 0.1

q = to_q(val,12)
ba = from_q(q,12)

print(f"0.1 -> Q12.12 = {q} -> กลับได้ {ba}")
print(f"คลาดเคลื่อน = {abs(val - ba):.8f}")

# บวกสะสม 1000 ครั้ง
acc = 0
for _ in range(1000):
    acc += q
print("บวก 0.1 จำนวน 1000 ครั้ง ควรได้ 100 แต่ได้:", from_q(acc,12))

0.1 -> Q12.12 = 410 -> กลับได้ 0.10009765625
คลาดเคลื่อน = 0.00009766
บวก 0.1 จำนวน 1000 ครั้ง ควรได้ 100 แต่ได้: 100.09765625


## 6. ตัวอย่างงานจริง: เซนเซอร์อุณหภูมิ
เซนเซอร์หลายรุ่นส่งค่ามาเป็นจำนวนเต็มที่ scale ไว้แล้ว

In [9]:
# เซนเซอร์ส่งค่าอุณหภูมิเป็น Q8.8 มาทาง I2C
raw_from_sensor = 6592        # 2 ไบต์ที่อ่านได้
temp = from_q(raw_from_sensor)
print(f"อุณหภูมิ = {temp} องศาเซลเซียส")

# อีกแบบที่พบบ่อย: เซนเซอร์คูณ 100 มาแล้ว (ไม่ใช่ฐานสอง)
raw2 = 2575
print(f"อุณหภูมิ = {raw2 / 100} องศาเซลเซียส")

อุณหภูมิ = 25.75 องศาเซลเซียส
อุณหภูมิ = 25.75 องศาเซลเซียส


### ลองเอง 4
เซนเซอร์ Q8.8 ส่งค่า `-2560` มา อุณหภูมิกี่องศา? (ระวังเรื่องเลขติดลบจากครั้งที่ 3)

In [10]:
print(from_q(-2560))   # ?

-10.0


## 7. เชื่อมไปภาษา C (ครั้งที่ 6)

| Python (วันนี้) | C บน ESP32 |
|---|---|
| `to_q(3.75)` | `(int16_t)(3.75 * 256)` |
| `from_q(v)` | `v / 256.0` *(หรือแสดงผลเป็นจำนวนเต็ม)* |
| `(a * b) >> N` | `(int16_t)(((int32_t)a * b) >> 8)` |

**ข้อควรระวังใน C:** ตอนคูณต้อง cast เป็น `int32_t` ก่อน ไม่งั้นผลคูณ overflow ตั้งแต่ยังไม่ทัน shift
(กลับไปดูเรื่อง overflow จากครั้งที่ 3 ได้เลย)

---
### สรุปสิ่งที่ต้องส่ง
1. ตอบคำถามท้ายกิจกรรม 5 ข้อ → โพสต์ reply ใน Teams
2. ทำ Quiz 10 ข้อบน Wayground
